# **IA para DEVs - 8IADT - Fase 3**

## **Tech Challenge - Assistente Medico Hospitalar**
### **Grupo 49**
### **Integrantes**:
- Rodrigo de Araújo Rosa
- Elias Maximiano da Silva
- Fábia Gomes de Jesus
- Danilo Pereira

### **Desafio**
Após o sucesso na automação de análises de exames e textos clínicos, o hospital quer avançar para um nível superior de personalização: criar um assistente virtual médico, treinado com os dados próprios do hospital, capaz de auxiliar nas condutas clínicas, responder dúvidas de médicos e sugerir procedimentos com base nos protocolos internos.

Além disso, a ideia é organizar fluxos de decisão automatizados e seguros, onde, por exemplo, ao receber informações sobre um paciente, o sistema possa acionar diferentes etapas, como verificar exames pendentes, sugerir tratamentos e emitir alertas para a equipe médica — tudo isso coordenado com LangChain.

# **Preprocessamento do Subdataset PQA-A**

Foi utilizado para o Subconjunto do dataset **PubMedQA** — uma coleção de pares de perguntas e respostas biomédicas derivadas de abstracts do PubMed.

O arquivo `ori_pqaa.json` corresponde às **211.269 instâncias geradas artificialmente** (subconjunto *PQA-A*, artificially generated) do dataset completo, que totaliza aproximadamente 273,5 mil instâncias entre exemplos rotulados por especialistas (1k), não rotulados (61,2k) e gerados artificialmente (211,3k).

- **Fonte**: [PubMedQA — A Dataset for Biomedical Research Question Answering](https://pubmedqa.github.io/)
- **Repositório oficial**: [github.com/pubmedqa/pubmedqa](https://github.com/pubmedqa/pubmedqa)
- **Formato**: JSON com objetos contendo os campos `QUESTION` e `LONG_ANSWER`, entre outros.
- **Idioma original**: Inglês
- **Uso neste projeto**: Entrada do pipeline de preprocessamento, onde os campos `QUESTION` e `LONG_ANSWER` são extraídos, traduzidos para Português Brasileiro, formatados em Alpaca e salvos no arquivo `train_data.json` para fine-tuning da LLM do Assistente Médico Hospitalar.

**Modelo base utilizado na tradução**: Helsinki-NLP/opus-mt-tc-big-en-pt  
**Ambiente**: Google Colab (GPU T4, 15 GB VRAM)  
**Idioma alvo**: Português Brasileiro  
**Tempo de execução**: 12 horas e 10 minutos


---

## **Indice**
1. Instalação de Dependências
2. Montagem do Google Drive
3. Preprocessamento e Tradução dos Dados

## 1. INSTALAÇÃO DE DEPENDÊNCIAS

In [5]:
%%time
# Instala todas as dependencias com versoes fixadas
!pip install -q \
    transformers==4.46.3 \
    peft==0.13.2 \
    accelerate==1.1.1 \
    datasets==3.1.0 \
    sacremoses==0.1.1 \
    sentencepiece==0.2.0 \
    python-dotenv==1.0.1 \
    langdetect==1.0.9 \
    protobuf
print("Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 40.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 115.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the 

In [6]:
%%time
import json
from typing import Optional
import torch
print("GPU disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
import transformers, peft
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

GPU disponivel: True
GPU: Tesla T4
Transformers: 4.46.3
PEFT: 0.13.2
CPU times: user 11.7 s, sys: 1.65 s, total: 13.4 s
Wall time: 18.9 s


## 2. MONTAGEM DO GOOGLE DRIVE

In [7]:
%%time
from google.colab import drive
import os
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/fase3-data"
RAW_DIR = os.path.join(BASE_DIR, "raw")
PREP_DIR = os.path.join(BASE_DIR, "preprocessed")
MODELS_DIR = os.path.join(BASE_DIR, "models")

for d in [RAW_DIR, PREP_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive montado. Diretorios criados.")
print("Base:", BASE_DIR)

Mounted at /content/drive
Drive montado. Diretorios criados.
Base: /content/drive/MyDrive/fase3-data
CPU times: user 530 ms, sys: 96.2 ms, total: 627 ms
Wall time: 21.6 s


## 3. PREPROCESSAMENTO E TRADUÇÃO DOS DADOS

In [8]:
INPUT_FILE = os.path.join(RAW_DIR, "ori_pqaa.json")
OUTPUT_FILENAME = "train_data.json"
print("Dataset encontrado:", os.path.isfile(INPUT_FILE))

Dataset encontrado: True


In [9]:
# Carrega o dataset do arquivo JSON
def load_dataset(path: str) -> list[dict]:
    with open(path, encoding="utf-8") as f:
            data = json.load(f)

    # Normaliza para lista de dicionários
    if isinstance(data, list):
        records = data
    elif isinstance(data, dict):
        records = list(data.values())
    else:
        print("Formato de dataset inválido. Esperado lista ou dicionário JSON.")

    return records

print("Função load_dataset() carregada.")

Função load_dataset() carregada.


In [10]:
# Verifica se um registro possui todos os campos de Pergunta (QUESTION) e Resposta (LONG_ANSWER)
def is_valid(record: dict) -> bool:
    if "QUESTION" in record and "LONG_ANSWER" in record:
        return True
    return False

# Filtra registros inválidos da lista fornecida
def filter_valid(records: list[dict]) -> tuple[list[dict], int]:
    if not isinstance(records, list):
        print("O argumento 'records' deve ser uma lista de dicionários.")

    valid: list[dict] = []
    discarded = 0

    for idx, record in enumerate(records):
        if is_valid(record):
            valid.append(record)
        else:
            discarded += 1
            print(f"Registro {idx} descartado: campos obrigatórios ausentes ou vazios.")

    return valid, discarded

print("Funções de validação carregadas.")

Funções de validação carregadas.


In [11]:
# ---------------------------------------------------------------------------
# Configurações
# ---------------------------------------------------------------------------

MODEL_NAME = "Helsinki-NLP/opus-mt-tc-big-en-pt"

# Tamanho do lote enviado à GPU por vez.
# Reduzir para 16 se ocorrer OOM na T4.
BATCH_SIZE = 32

# Limite de tokens por texto. O modelo aceita até 512; usamos 480 para folga.
# Textos maiores são truncados ANTES de entrar no pipeline.
MAX_TOKENS = 480

# Nome do arquivo de checkpoint
CHECKPOINT_FILENAME = "translation_checkpoint.json"

# Salvar checkpoint a cada N registros totais traduzidos
CHECKPOINT_EVERY = 1000

# ---------------------------------------------------------------------------
# Carregamento do modelo (lazy, executado uma única vez)
# ---------------------------------------------------------------------------

_pipeline = None

# Carrega o pipeline de tradução na GPU (se disponível) ou CPU
def load_model() -> None:
    global _pipeline
    if _pipeline is not None:
        return  # Já carregado

    from transformers import pipeline as hf_pipeline
    import torch

    device = 0 if torch.cuda.is_available() else -1  # 0 = primeira GPU, -1 = CPU
    device_label = "GPU (CUDA)" if device == 0 else "CPU"
    print(f"Carregando modelo '{MODEL_NAME}' em {device_label}...")

    _pipeline = hf_pipeline(
        "translation",
        model=MODEL_NAME,
        device=device,
        max_length=MAX_TOKENS,
    )
    print("Modelo carregado com sucesso.")


# ---------------------------------------------------------------------------
# Tradução
# ---------------------------------------------------------------------------

# Trunca o texto a um número máximo de caracteres antes de enviar ao pipeline
def _truncate(text: str, max_chars: int = MAX_TOKENS * 4) -> str:
    return text[:max_chars] if len(text) > max_chars else text

# Traduz uma lista de textos EN→PT-BR usando o pipeline carregado
def translate_batch(texts: list[str]) -> list[str]:
    if not texts:
        return []

    truncated = [_truncate(t) for t in texts]

    try:
        results = _pipeline(truncated, batch_size=BATCH_SIZE)

        # Extrai o texto traduzido de cada resultado
        translated = []
        for r in results:
            if isinstance(r, list):
                translated.append(r[0].get("translation_text", ""))
            else:
                translated.append(r.get("translation_text", ""))

        # Validação crítica: o pipeline DEVE retornar um item por input
        if len(translated) != len(texts):
            raise ValueError(
                f"Pipeline retornou {len(translated)} itens para {len(texts)} entradas. "
                "Ativando fallback item a item."
            )

        return translated

    except Exception as exc:
        print(f"[AVISO] Falha no batch: {exc}")
        print("  Retraduzindo item a item para recuperar o lote...")
        return _translate_one_by_one(truncated)

# Traduz cada texto individualmente. Usado como fallback quando o batch falha
def _translate_one_by_one(texts: list[str]) -> list[str]:
    results = []
    for i, text in enumerate(texts):
        try:
            r = _pipeline(text, batch_size=1)
            if isinstance(r, list) and isinstance(r[0], dict):
                results.append(r[0].get("translation_text", ""))
            elif isinstance(r, list) and isinstance(r[0], list):
                results.append(r[0][0].get("translation_text", ""))
            else:
                results.append("")
        except Exception as exc:
            print(f"  [ERRO] Item {i} falhou individualmente: {exc}. Inserindo string vazia.")
            results.append("")
    return results


# ---------------------------------------------------------------------------
# Checkpoint
# ---------------------------------------------------------------------------

# Salva o progresso em disco de forma atômica (via arquivo temporário)
def save_checkpoint(path: str, records: list[dict]) -> None:
    try:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        tmp = path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump({"translated_count": len(records), "records": records}, f, ensure_ascii=False)
        os.replace(tmp, path)
        print(f"  [Checkpoint] {len(records)} registros salvos em '{path}'.")
    except OSError as exc:
        print(f"  [AVISO] Falha ao salvar checkpoint: {exc}")

# Tenta carregar registros de um checkpoint existente. Retorna lista vazia em caso de falha
def load_checkpoint(path: str) -> list[dict]:
    if not os.path.isfile(path):
        return []
    try:
        with open(path, encoding="utf-8") as f:
            ckpt = json.load(f)
        records = ckpt.get("records", [])
        print(f"[Checkpoint] Encontrado: {len(records)} registros já traduzidos. Retomando...")
        return records
    except (json.JSONDecodeError, KeyError, OSError) as exc:
        print(f"[AVISO] Checkpoint corrompido ('{exc}'). Iniciando do zero.")
        return []

# ---------------------------------------------------------------------------
# Função principal de tradução
# ---------------------------------------------------------------------------

# Traduz todos os registros EN→PT-BR com suporte a checkpoint
def translate_all(
    records: list[dict],
    checkpoint_dir: Optional[str] = None,
    question_field: str = "QUESTION",
    answer_field: str = "LONG_ANSWER",
) -> list[dict]:
    load_model()

    checkpoint_path = (
        os.path.join(checkpoint_dir, CHECKPOINT_FILENAME)
        if checkpoint_dir
        else None
    )

    # Retoma do checkpoint, se existir
    translated_records = load_checkpoint(checkpoint_path) if checkpoint_path else []
    start_index = len(translated_records)
    remaining = records[start_index:]
    total = len(records)

    if not remaining:
        print(f"Todos os {total} registros já foram traduzidos.")
        return translated_records

    print(f"Traduzindo {len(remaining)} registros restantes (total do dataset: {total})...")

    records_since_save = 0

    try:
        for batch_start in range(0, len(remaining), BATCH_SIZE):
            batch = remaining[batch_start: batch_start + BATCH_SIZE]

            questions = [r.get(question_field, "") for r in batch]
            answers   = [r.get(answer_field,   "") for r in batch]

            translated_questions = translate_batch(questions)
            translated_answers   = translate_batch(answers)

            for i, record in enumerate(batch):
                new_record = dict(record)
                new_record[question_field] = translated_questions[i]
                new_record[answer_field]   = translated_answers[i]
                translated_records.append(new_record)

            records_since_save += len(batch)
            done = len(translated_records)
            pct  = 100.0 * done / total
            print(f"Progresso: {done}/{total} ({pct:.2f}%)")

            # Checkpoint intermediário
            if checkpoint_path and records_since_save >= CHECKPOINT_EVERY:
                save_checkpoint(checkpoint_path, translated_records)
                records_since_save = 0

        # Checkpoint final
        if checkpoint_path:
            save_checkpoint(checkpoint_path, translated_records)

    except KeyboardInterrupt:
        done = len(translated_records)
        print(f"\nInterrompido manualmente. {done}/{total} registros traduzidos.")
        if checkpoint_path and done > start_index:
            save_checkpoint(checkpoint_path, translated_records)
        print("Execute novamente para retomar do ponto salvo.")

    print(f"Tradução concluída: {len(translated_records)}/{total} registros.")
    return translated_records

print("Funções de tradução carregadas.")

Funções de tradução carregadas.


In [12]:
# Salva os registros preprocessados no formato Alpaca (instruction/input/output)
def save_preprocessed(records: list[dict], output_dir: str) -> str:
    try:
        os.makedirs(output_dir, exist_ok=True)
    except OSError as exc:
        print(f"Não foi possível criar o diretório de saída '{output_dir}'")

    output_path = os.path.join(output_dir, OUTPUT_FILENAME)

    # Converte para formato Alpaca
    alpaca_records = [
        {
            "instruction": record.get("QUESTION", ""),
            "input": "",
            "output": record.get("LONG_ANSWER", ""),
        }
        for record in records
    ]

    try:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(alpaca_records, f, ensure_ascii=False, indent=2)
    except OSError as exc:
        print(f"Erro ao salvar o arquivo preprocessado '{output_path}'")

    print(f"Arquivo salvo: {output_path} ({len(alpaca_records)}).")
    return output_path

print("Função de salvamento carregada.")

Função de salvamento carregada.


In [ ]:
%%time
Pipeline: carregar -> traduzir ->  salvar
if os.path.isfile(INPUT_FILE):

    print("Iniciando preprocessamento...")
    print("A traducao pode levar até 12 horas dependendo do tamanho do dataset.")

    # 1. Carregar dataset
    raw_records = load_dataset(INPUT_FILE)
    total_original = len(raw_records)
    print(f"Dataset carregado: {total_original} registros.")

    # 2. Validar e filtrar registros com campos ausentes
    valid_records, discarded_missing = filter_valid(raw_records)
    total_discarded_missing_fields = discarded_missing
    print(f"Validação: {len(valid_records)} válidos, {discarded_missing} descartados (campos ausentes).")

    # 3. Traduzir dataset EN→PT-BR
    translated_records = translate_all(valid_records,PREP_DIR)
    total_translated = len(translated_records)

    # 4. Salvar novo dataset preparado para treinamento
    total_used_for_training = len(translated_records)
    output_path = save_preprocessed(translated_records, PREP_DIR)
    print(f"Dados salvos em: {PREP_DIR}")

    print("Preprocessamento concluido.")

else:
    print("AVISO: Dataset nao encontrado em", INPUT_FILE)

Iniciando preprocessamento...
A traducao pode levar 10-30 minutos dependendo do tamanho do dataset.
Dataset carregado: 211269 registros.
Validação: 211269 válidos, 0 descartados (campos ausentes).
Carregando modelo 'Helsinki-NLP/opus-mt-tc-big-en-pt' em GPU (CUDA)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/465M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

...

Progresso: 109440/211269 (51.80%)
Progresso: 109472/211269 (51.82%)
Progresso: 109504/211269 (51.83%)
Progresso: 109536/211269 (51.85%)
Progresso: 109568/211269 (51.86%)
  [Checkpoint] 109568 registros salvos em '/content/drive/MyDrive/fase3-data/preprocessed/translation_checkpoint.json'.
Progresso: 109600/211269 (51.88%)
Progresso: 109632/211269 (51.89%)
Progresso: 109664/211269 (51.91%)
Progresso: 109696/211269 (51.92%)
Progresso: 109728/211269 (51.94%)
Progresso: 109760/211269 (51.95%)
Progresso: 109792/211269 (51.97%)
Progresso: 109824/211269 (51.98%)
Progresso: 109856/211269 (52.00%)
Progresso: 109888/211269 (52.01%)
Progresso: 109920/211269 (52.03%)
Progresso: 109952/211269 (52.04%)
Progresso: 109984/211269 (52.06%)
Progresso: 110016/211269 (52.07%)
Progresso: 110048/211269 (52.09%)
Progresso: 110080/211269 (52.10%)
Progresso: 110112/211269 (52.12%)
Progresso: 110144/211269 (52.13%)
Progresso: 110176/211269 (52.15%)
Progresso: 110208/211269 (52.16%)
Progresso: 110240/211269 (52.1

Your input_length: 437 is bigger than 0.9 * max_length: 480. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Progresso: 152768/211269 (72.31%)
Progresso: 152800/211269 (72.32%)
Progresso: 152832/211269 (72.34%)
Progresso: 152864/211269 (72.36%)
Progresso: 152896/211269 (72.37%)
Progresso: 152928/211269 (72.39%)
Progresso: 152960/211269 (72.40%)
Progresso: 152992/211269 (72.42%)
Progresso: 153024/211269 (72.43%)
Progresso: 153056/211269 (72.45%)
Progresso: 153088/211269 (72.46%)
Progresso: 153120/211269 (72.48%)
Progresso: 153152/211269 (72.49%)
Progresso: 153184/211269 (72.51%)
Progresso: 153216/211269 (72.52%)
Progresso: 153248/211269 (72.54%)
Progresso: 153280/211269 (72.55%)
Progresso: 153312/211269 (72.57%)
Progresso: 153344/211269 (72.58%)
Progresso: 153376/211269 (72.60%)
Progresso: 153408/211269 (72.61%)
Progresso: 153440/211269 (72.63%)
Progresso: 153472/211269 (72.64%)
Progresso: 153504/211269 (72.66%)
Progresso: 153536/211269 (72.67%)
Progresso: 153568/211269 (72.69%)
Progresso: 153600/211269 (72.70%)
  [Checkpoint] 153600 registros salvos em '/content/drive/MyDrive/fase3-data/prepr

Your input_length: 464 is bigger than 0.9 * max_length: 480. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Progresso: 197216/211269 (93.35%)
Progresso: 197248/211269 (93.36%)
Progresso: 197280/211269 (93.38%)
Progresso: 197312/211269 (93.39%)
Progresso: 197344/211269 (93.41%)
Progresso: 197376/211269 (93.42%)
Progresso: 197408/211269 (93.44%)
Progresso: 197440/211269 (93.45%)
Progresso: 197472/211269 (93.47%)
Progresso: 197504/211269 (93.48%)
Progresso: 197536/211269 (93.50%)
Progresso: 197568/211269 (93.51%)
Progresso: 197600/211269 (93.53%)
Progresso: 197632/211269 (93.55%)
  [Checkpoint] 197632 registros salvos em '/content/drive/MyDrive/fase3-data/preprocessed/translation_checkpoint.json'.
Progresso: 197664/211269 (93.56%)
Progresso: 197696/211269 (93.58%)
Progresso: 197728/211269 (93.59%)
Progresso: 197760/211269 (93.61%)
Progresso: 197792/211269 (93.62%)
Progresso: 197824/211269 (93.64%)
Progresso: 197856/211269 (93.65%)
Progresso: 197888/211269 (93.67%)
Progresso: 197920/211269 (93.68%)
Progresso: 197952/211269 (93.70%)
Progresso: 197984/211269 (93.71%)
Progresso: 198016/211269 (93.7

In [ ]:
# Exibe relatorio
print("=" * 50)
print("RELATORIO")
print("=" * 50)
print("Total original:         ", total_original)
print("Descartados (ausentes): ", total_discarded_missing_fields)
print("Traduzidos:             ", total_translated)
print("Usados no treinamento:  ", total_used_for_training)

RELATORIO
Total original:          211269
Descartados (ausentes):  0
Traduzidos:              211269
Usados no treinamento:   211269
